<a href="https://colab.research.google.com/github/hbprosper/dihiggs/blob/main/code/heft_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Higgs Effective Field Theory (HEFT) Study: Huber-Pull Fine-Tune
> Created: Oct 31, 2026 Harrison B. Prosper and Claude<br>

## Introduction

The L1 loss of `heft_train.ipynb` treats an error of $10^{-4}$ pb the same in every bin, so the fit is dominated by the large cross sections and the region just above threshold is poorly modelled (mean $\chi^2$ per bin 8.4 at 255–555 GeV on the holdout spectra). The BNN runs showed that measuring each error in units of the bin's POWHEG noise, starting from the L1 solution and using gradients over all the training rows, brings that down to about 1.9. This notebook tests the same idea as an ordinary training step, so that it can be used for every member of an ensemble. The plan is `docs/huber/huber_plan.pdf`.

### The loss

For a training row with POWHEG value $t_k$, prediction $f_k$ and noise variance $\sigma_k^2$, the **pull** is $r_k = (f_k - t_k)/\sigma_k$, and the loss is the average of the **Huber** function
$$
\ell_\delta(r) = \begin{cases} \tfrac12 r^2, & |r| \le \delta,\\ \delta\,(|r| - \tfrac12\delta), & |r| > \delta. \end{cases}
$$
For bins within $\delta$ standard deviations of POWHEG this is $\chi^2/2$; bins further away contribute only linearly, so a few badly modelled bins cannot dominate the fit. $\delta = $ `None` gives plain $\chi^2/2$.

### The noise

Each POWHEG spectrum behaves as a histogram of about $N_\text{eff}$ equal-weight events, estimated from the bin-to-bin jitter of the training spectra, and each bin is given the variance of one extra event:
$$
\sigma_k^2 = \Big(t_k + \frac{\sigma_\text{tot}}{N_\text{eff}}\Big)\frac{\sigma_\text{tot}}{N_\text{eff}} ,
$$
where $\sigma_\text{tot}$ is the total cross section of the spectrum. The variances are **not** a target: the target is the cross section alone, and the variances are a separate tensor, aligned with the rows, handed to the loss beside the targets. With `VARIANCE_FROM = 'model'`, $t_k$ in this formula is replaced by the model's prediction (held fixed during each step).

### Two stages

1. **Stage 1:** the L1 training of `heft_train.ipynb` (here: the existing model `START`).
2. **Stage 2:** the fine-tune: Adam with **full-batch** gradients (every step uses all the training rows), a small learning rate halved at regular intervals, and the parameters with the lowest validation loss kept.

One fine-tune is run for each value of $\delta$ in `DELTAS`, each from the same starting model, and the results are compared on the 1,000 holdout spectra.

In [ ]:
# ------------------------------------------------------------------
# Google Colab setup (nothing happens when run outside Colab)
#   1. clone the dihiggs repository, which holds the data files,
#   2. install the dihiggs package, which also installs drwho,
#   3. make dihiggs/code the working directory so that the relative
#      paths used below (../data, runs/) work as they do locally.
#
# Use a GPU: in Colab choose Runtime > Change runtime type > GPU.
# ------------------------------------------------------------------
import os, sys, subprocess, importlib

IN_COLAB = 'google.colab' in sys.modules

# With USE_DRIVE = True the runs/ folder lives in your Google Drive
# (MyDrive/dihiggs/runs), where the starting model trained by
# heft_train.ipynb is found.
USE_DRIVE = True

if IN_COLAB:
    REPO = '/content/dihiggs'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q',
                        'https://github.com/hbprosper/dihiggs.git', REPO],
                       check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', REPO],
                   check=True)
    importlib.invalidate_caches()
    os.chdir(f'{REPO}/code')

    if USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_runs = '/content/drive/MyDrive/dihiggs/runs'
        os.makedirs(drive_runs, exist_ok=True)
        if not os.path.exists('runs'):
            os.symlink(drive_runs, 'runs')
        print(f'runs/ is stored in {drive_runs}')

print(f'working directory: {os.getcwd()}')

In [ ]:
import os, sys, time

# the standard module for tabular data
import pandas as pd

# the standard module for array manipulation
import numpy as np

# the standard modules for high-quality plots
import matplotlib as mp
import matplotlib.pyplot as plt

# pytorch
import torch

# module for shell utilities
import shutil

# ML and other utilities
import drwho.nn as mlp

# the fine-tune: noise model, Huber-pull loss, full-batch training
from dihiggs import finetune as ft

# update fonts
plt.rcParams.update({
  "text.usetex": shutil.which('latex') is not None,
  "font.family": "sans-serif",
  "font.size": 12
})

## Computational device

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\n\tAvailable device: {str(DEVICE):4s}\n')

## Load training data

In [ ]:
datafile = f'../data/heft_wide_gauss_traindata_13TeV.csv.gz'

print(f'loading {datafile}')
df = pd.read_csv(datafile)

sample_size = len(df)

# Fingerprint of the training file, saved in each run's configuration.
import hashlib
with open(datafile, 'rb') as f:
    traindata_sha256 = hashlib.sha256(f.read()).hexdigest()
print(f'sample size: {sample_size}')

## Switches

| switch | meaning |
| :--- | :--- |
| `START` | the trained L1 model (stage 1) to fine-tune |
| `DELTAS` | the values of the Huber threshold $\delta$, in units of the noise; `None` means plain $\chi^2$ |
| `VARIANCE_FROM` | `'data'`: $\sigma_k$ from the POWHEG value; `'model'`: from the model's prediction |
| `FT_STEPS` | number of full-batch steps |
| `FT_LR` | initial learning rate; it is halved 3 times, at equal intervals |
| `CHUNK_SIZE` | rows per chunk when the full-batch gradient is accumulated; lower it if the GPU runs out of memory |
| `REDO` | if False, a fine-tune whose folder already holds parameters is not repeated |

In [ ]:
# ------------------------------------------------------------------
# Switches
# ------------------------------------------------------------------
START         = 'runs/L1Loss/dihiggs_params.pth'
DELTAS        = [1, 2, 3, None]       # None: plain chi-squared
VARIANCE_FROM = 'data'                # 'data' or 'model'
FT_STEPS      = 5000
FT_LR         = 1e-4
CHUNK_SIZE    = 2**16
SEED          = 42
REDO          = False

assert VARIANCE_FROM in ('data', 'model'), VARIANCE_FROM

NAME = 'dihiggs'

def dirname(delta):
    # the run folder for one value of delta, e.g. FT_huber_d2 or FT_chi2;
    # with VARIANCE_FROM = 'model' the name ends in _pearson
    name = 'FT_chi2' if delta is None else f'FT_huber_d{delta:g}'
    return name + ('_pearson' if VARIANCE_FROM == 'model' else '')

print('run folders:', [f'runs/{dirname(d)}' for d in DELTAS])

## The split, the noise and the tensors

The split of `heft_train.ipynb`: the first 86.4% of the rows for training, the next 1.6% for validation, the last 12% as the quick test (not used here). $N_\text{eff}$ is estimated from the spectra in the training file, as in `heft_train.ipynb`, and the noise variance of every row is computed once.

In [ ]:
from dihiggs import heftnet as NN

train_size = int(0.864*sample_size)
test_size  = int(0.120*sample_size)
val_size   = sample_size - train_size - test_size

N_EFF = ft.estimate_n_eff(df)
var, unit = ft.noise_variance(df, N_EFF)
print(f'effective number of events per spectrum: N_eff = {N_EFF:.0f}')

def tensors(rows):
    # inputs [n, 6], targets [n], variances [n], one-event units [n], on
    # the device; the four tensors are aligned row by row
    X = torch.tensor(df[NN.FEATURES].to_numpy(np.float32)[rows], device=DEVICE)
    T = torch.tensor(df[NN.TARGET].to_numpy(np.float32)[rows], device=DEVICE)
    V = torch.tensor(var[rows].astype(np.float32), device=DEVICE)
    U = torch.tensor(unit[rows].astype(np.float32), device=DEVICE)
    return X, T, V, U

X_tr, T_tr, V_tr, U_tr = tensors(slice(0, train_size))
X_va, T_va, V_va, U_va = tensors(slice(train_size, train_size + val_size))
print(f'train: {len(X_tr)}   val: {len(X_va)}')

## The fine-tunes

For each $\delta$: rebuild the model, load `START`, fine-tune, and save in `runs/<dirname>/`, an ordinary run folder (`dihiggs_config.yaml`, `dihiggs_params.pth` with the lowest validation loss, `dihiggs_losses.csv`), so that `heft_test.ipynb` and `heft_export.ipynb` work with it. The starting model counts as the first "best", so a fine-tune can never end worse, on the validation rows, than where it started.

In [ ]:
def run_one(delta):
    folder = dirname(delta)
    params = f'runs/{folder}/{NAME}_params.pth'
    if os.path.exists(params) and not REDO:
        print(f'{folder}: already done (set REDO = True to repeat)')
        return
    t0 = time.time()
    torch.manual_seed(SEED)

    config = mlp.Config(NAME, dirname=folder)
    config('model_name', NN.NAME)
    config('features', NN.FEATURES)
    config('target', NN.TARGET)
    config('activation', NN.ACTIVATION)
    config('traindata_sha256', traindata_sha256)
    config('train_size', train_size)
    config('val_size', val_size)
    config('test_size', test_size)
    config('loss', 'huber_pull' if delta is not None else 'chi2')
    config('finetune/start', START)
    config('finetune/delta', -1 if delta is None else float(delta))
    config('finetune/variance_from', VARIANCE_FROM)
    config('finetune/N_eff', float(N_EFF))
    config('finetune/steps', FT_STEPS)
    config('finetune/lr', FT_LR)
    config.save()

    model = NN.HEFTNet()
    model.load(START)            # drwho's Model.load: onto the CPU
    model.to(DEVICE)

    print(f'\n--- {folder} ---')
    ft.finetune(model, X_tr, T_tr, V_tr, U_tr, X_va, T_va, V_va, U_va,
                ft.HuberPull(delta), steps=FT_STEPS, lr=FT_LR,
                variance_from=VARIANCE_FROM, chunk_size=CHUNK_SIZE,
                paramsfile=config('file/params'),
                lossfile=config('file/losses'))
    print(f'{folder}: {(time.time() - t0) / 60:.1f} min')

for delta in DELTAS:
    run_one(delta)

### Loss curves

The training and validation losses of each fine-tune (each in its own units, since the loss depends on $\delta$). They should fall and level off; a validation loss that rises while the training loss falls would mean overfitting.

In [ ]:
fig, axes = plt.subplots(1, len(DELTAS), figsize=(4 * len(DELTAS), 3.5))
axes = np.atleast_1d(axes)
for ax, delta in zip(axes, DELTAS):
    h = pd.read_csv(f'runs/{dirname(delta)}/{NAME}_losses.csv')
    ax.plot(h.step, h.train, label='train')
    ax.plot(h.step, h.val, label='validation')
    ax.set_yscale('log')
    ax.set_xlabel('step')
    ax.set_title(dirname(delta), fontsize=11)
axes[0].set_ylabel('loss')
axes[0].legend(fontsize=10)
fig.tight_layout()
plt.savefig(f'runs/{dirname(DELTAS[0])}/{NAME}_finetune_losses.png')
plt.show()

## Results on the holdout spectra

For the starting L1 model, every fine-tuned model and, if its folder exists, the full-data BNN (`BNN_DIR`, its posterior mean): the median $\chi^2/N$ per spectrum, the fractions of spectra with $\chi^2/N$ above 2 and 5, the mean $\chi^2$ per bin below and above 555 GeV, the mean absolute residual (what L1 optimises), and the fraction of negative predictions. As in `heft_test.ipynb`, the noise variance of a holdout bin is $t\,\sigma_\text{tot}/N_\text{eff}$.

In [ ]:
BNN_DIR = 'runs/BNN_poisson_full'

spectra = pd.read_csv('../data/heft_wide_gauss_testdata_13TeV.csv')
params  = ['klambda', 'CT', 'CTT', 'CGGH', 'CGGHH']
bins    = [c for c in spectra.columns if c.isdigit()]
T_hold  = spectra[bins].to_numpy(np.float64)          # [1000, 80] POWHEG
n_spec, n_bins = T_hold.shape
x_bins  = (np.arange(n_bins) + 0.5) / 100             # bin centres in x
mhh     = 255 + 1500 * x_bins                         # in GeV
theta   = spectra[params].to_numpy(np.float32)
inputs  = np.column_stack([np.tile(x_bins, n_spec).astype(np.float32),
                           np.repeat(theta, n_bins, axis=0)])
x_hold  = torch.from_numpy(inputs).to(DEVICE)
T_tot   = T_hold.sum(axis=1, keepdims=True)
noise_hold = np.where(T_hold > 0, T_hold * T_tot / N_EFF, np.nan)

def predict(paramsfile):
    model = NN.HEFTNet()
    model.load(paramsfile)
    model.to(DEVICE)
    with torch.no_grad():
        return model(x_hold).double().cpu().numpy().reshape(n_spec, n_bins)

predictions = {'L1 (start)': predict(START)}
for delta in DELTAS:
    predictions[dirname(delta)] = predict(f'runs/{dirname(delta)}/{NAME}_params.pth')
if os.path.exists(BNN_DIR):
    from dihiggs import hmc
    bnn = hmc.BNN(NN.HEFTNet, BNN_DIR, device=DEVICE, max_samples=200)
    predictions['BNN, full data (mean)'] = \
        bnn.mean(x_hold).cpu().numpy().reshape(n_spec, n_bins)

chi2 = {}
print(f'{"":24s} {"med chi2/N":>10s} {">2":>6s} {">5":>6s} {"255-555":>8s} '
      f'{">555":>6s} {"|res| (pb)":>11s} {"neg.":>8s}')
for name, f in predictions.items():
    c = (f - T_hold)**2 / noise_hold                  # chi2 of every bin
    chi2[name] = c
    per_spectrum = np.nanmean(c, axis=1)
    print(f'{name:24s} {np.median(per_spectrum):10.3f} '
          f'{np.mean(per_spectrum > 2):6.1%} {np.mean(per_spectrum > 5):6.1%} '
          f'{np.nanmean(c[:, :20]):8.2f} {np.nanmean(c[:, 20:]):6.3f} '
          f'{np.mean(np.abs(f - T_hold)):11.3e} {np.mean(f < 0):8.3%}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name, c in chi2.items():
    ls = '--' if name.startswith('BNN') else '-'
    ax.plot(mhh, np.nanmean(c, axis=0), ls, label=name)
ax.axhline(1, color='gray', ls=':')
ax.set_yscale('log')
ax.set_xlabel(r'$m_{hh}$ (GeV)')
ax.set_ylabel(r'mean $\chi^2$ per bin')
ax.legend(fontsize=10)
fig.tight_layout()
plt.savefig(f'runs/{dirname(DELTAS[0])}/{NAME}_finetune_chi2.png')
plt.show()

### What counts as success

A value of $\delta$ for which the mean $\chi^2$ per bin at 255–555 GeV is close to the BNN's (about 1.9), the $\chi^2$ above 555 GeV is no worse than the L1 model's (1.20), and negative predictions are fewer. That $\delta$ is then used for every member of the ensemble (`heft_train_ensemble.ipynb`, after this test).